# Chapter 8 — Build It Yourself: Devices & the GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ImAlno/Understanding-LLMs/blob/main/chapters/08-speed-device/code/explore.ipynb)

Detect your device, move tensors and a model onto it, meet the same-device rule, benchmark CPU vs
GPU, and train on the device. **On Colab: Runtime → Change runtime type → GPU, then Run all.** It
also runs on a CPU (no speedup) or an Apple GPU. "✍️ Your turn", "▶️ Run this", "▶️ Check your work". ⚡

> ⚠️ **Heads-up:** the two **✍️ Your turn** cells (Steps 2 and 5) start *unfinished*. If you "Run
> all" before filling them in, their check prints a **⚠️ fill-this-in** note — that's the cue to
> edit the cell, **not** a bug. The **▶️ Run this** cells run as-is.

## Step 1 — What are we running on?  ▶️ Run this

In [ ]:
import torch, time
import torch.nn as nn
import torch.nn.functional as F

def get_device():
    if torch.cuda.is_available():           # NVIDIA GPU (Colab, real training)
        return "cuda"
    if torch.backends.mps.is_available():   # Apple Silicon GPU
        return "mps"
    return "cpu"

def sync(device):                           # wait for the GPU to actually finish (for honest timing)
    if device == "cuda": torch.cuda.synchronize()
    elif device == "mps": torch.mps.synchronize()

device = get_device()
print(f"running on: {device}")
if device == "cpu":
    print("(no GPU detected — on Colab do Runtime → Change runtime type → GPU, then Run all)")

## Step 2 — Put data and model on the device  ✍️ Your turn

Move both the tensor `x` and the `model` onto `device` with `.to(device)`.

<details><summary>Stuck? Show the answer</summary>

```python
x = x.to(device)
model = model.to(device)
```
</details>

In [ ]:
x = torch.randn(1000, 1000)
model = nn.Sequential(nn.Linear(1000, 256), nn.ReLU(), nn.Linear(256, 10))

# ✍️ put BOTH the tensor and the model onto `device`
x = x            # replace
model = model    # replace

print("x is on:", x.device, "| model is on:", next(model.parameters()).device)

In [ ]:
# ▶️ Check your work
try:
    want = torch.device(device).type
    on_device = (x.device.type == want and next(model.parameters()).device.type == want)
    if device == "cpu":
        print("⚠️  Running on CPU, where .to(device) is a no-op — this can't tell whether you filled "
              "the cell in. On Colab use Runtime → GPU to really exercise it.")
    elif not on_device:
        print("⚠️  Not on the device yet — fill in the ✍️ cell above (move x and model with .to(device)) "
              "and re-run. (Expected until you do — not a bug.)")
    else:
        assert model(x).shape == (1000, 10)
        print(f"✅ Data and model are both on '{device}' — the device-agnostic pattern.")
except Exception as e:
    print("❌", e)

## Step 3 — The one rule: same device  ▶️ Run this

Adding tensors on different devices is an error (the #1 GPU bug). The fix: move the stray one.

In [ ]:
a_cpu = torch.randn(3)                  # on the CPU
b_dev = torch.randn(3, device=device)   # on `device`
try:
    (a_cpu + b_dev)
    print("a_cpu + b_dev worked — you're on CPU, so there's only one device here.")
except RuntimeError as e:
    print("RuntimeError (expected with a GPU):", str(e).split(chr(10))[0][:70], "...")
    print("the fix — move the stray tensor onto the device:", (a_cpu.to(device) + b_dev).device)

In [ ]:
# ▶️ Check your work
try:
    fixed = a_cpu.to(device) + b_dev
    assert fixed.device.type == torch.device(device).type
    print("✅ Same-device rule: move the stray tensor with .to(device) — in training, each batch.")
except Exception as e:
    print("❌", e)

## Step 4 — The speedup, the async trap, and the crossover  ▶️ Run this

First the async trap (timing without `synchronize` lies), then the CPU-vs-GPU crossover.

In [ ]:
def bench(dev, n, reps=10):
    a = torch.randn(n, n, device=dev); b = torch.randn(n, n, device=dev)
    a @ b; sync(dev)                                  # warmup
    t0 = time.perf_counter()
    for _ in range(reps): c = a @ b
    sync(dev)                                         # wait for the GPU before stopping the clock
    return (time.perf_counter() - t0) / reps * 1000

if device != "cpu":
    a = torch.randn(4096, 4096, device=device); b = torch.randn(4096, 4096, device=device)
    a @ b; sync(device)
    t0 = time.perf_counter(); a @ b; no_sync = (time.perf_counter() - t0) * 1000
    t0 = time.perf_counter(); a @ b; sync(device); yes_sync = (time.perf_counter() - t0) * 1000
    print(f"4096 matmul timed WITHOUT synchronize: {no_sync:6.2f} ms  (a lie — just the launch)")
    print(f"4096 matmul timed WITH    synchronize: {yes_sync:6.2f} ms  (the honest time)\n")

print(f"{'size':>6} {'CPU ms':>9} {'GPU ms':>9} {'speedup':>8}")
speedups = {}
for n in [256, 1024, 4096]:
    ct = bench("cpu", n)
    if device != "cpu":
        gt = bench(device, n); speedups[n] = ct / gt
        print(f"{n:>6} {ct:>9.2f} {gt:>9.2f} {ct / gt:>6.1f}x")
    else:
        print(f"{n:>6} {ct:>9.2f} {'—':>9} {'—':>8}")

In [ ]:
# ▶️ Check your work
try:
    if device != "cpu":
        assert speedups[256] < speedups[4096], "the speedup should grow with size (the crossover)"
        print(f"✅ The GPU's advantage GROWS with size: {speedups[256]:.1f}x at 256 → {speedups[4096]:.1f}x at 4096. "
              f"(A weak laptop GPU even loses on small matmuls; a datacenter GPU like Colab's wins at every size here.)")
    else:
        print("✅ Ran on CPU (no GPU column). Run on Colab with a GPU to see the speedup & crossover.")
except Exception as e:
    print("❌", e)

## Step 5 — Train on the device  ✍️ Your turn

Move the data (`Xtr`, `ytr`) and the model (`net`) onto `device`, then the same training loop you
know runs there.

<details><summary>Stuck? Show the answer</summary>

```python
Xtr = Xtr.to(device)
ytr = ytr.to(device)
net = net.to(device)
```
</details>

In [ ]:
torch.manual_seed(0)
Xtr = torch.randn(2000, 20)
ytr = Xtr @ torch.randn(20, 1)
net = nn.Sequential(nn.Linear(20, 128), nn.ReLU(), nn.Linear(128, 1))

# ✍️ move the data AND the model onto `device`
Xtr = Xtr        # replace
ytr = ytr        # replace
net = net        # replace

opt = torch.optim.AdamW(net.parameters(), lr=1e-2)
first = loss = None
try:
    for step_i in range(400):
        loss = F.mse_loss(net(Xtr), ytr)
        first = loss.item() if first is None else first
        opt.zero_grad(); loss.backward(); opt.step()
    print(f"loss {first:.3f} -> {loss.item():.4f}, all on '{net[0].weight.device.type}'")
except RuntimeError as e:
    print("⚠️ device mismatch — Xtr, ytr, AND net must all be on `device`:", str(e).split(chr(10))[0][:55])
    loss = None

In [ ]:
# ▶️ Check your work
try:
    want = torch.device(device).type
    if device == "cpu":
        print("⚠️  Running on CPU, where .to(device) is a no-op — this can't verify your fill. "
              "On Colab use Runtime → GPU.")
    elif loss is None:
        print("⚠️  Training didn't run — make sure Xtr, ytr, AND net are all on `device`, then re-run.")
    elif next(net.parameters()).device.type != want:
        print("⚠️  Not on the device yet — move net (and the data) with .to(device), then re-run. (Not a bug.)")
    else:
        assert loss.item() < first
        print(f"✅ Trained on '{device}' — loss {first:.2f} -> {loss.item():.4f}. The same loop runs on any device.")
except Exception as e:
    print("❌", e)

## 🎉 You can use a GPU now.

`device = get_device()`, `.to(device)` on the model and the data, keep everything on one device,
and `synchronize` before you time — that's the whole game. The *same* GPT from Chapter 5 trains on
a GPU many times faster with exactly these changes. Next: the [exercises](../exercises/) and the
[mini-project](../project/), *Benchmark Your GPU* (run it on Colab for the real numbers). 👋